# Neural Collaborative Filtering


### 1. Imports

In [1]:
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import sqrt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MultiLabelBinarizer

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### 2. Load Dataset

In [3]:
keywords = pd.read_csv("../data/keywords.csv")
ratings = pd.read_csv("../data/ratings_small.csv")
movies = pd.read_csv("../data/movies_metadata.csv", low_memory=False)
links = pd.read_csv("../data/links.csv")

print("Keywords:")
print("\n")
keywords.info()
keywords.head()
print("\n")

print("Ratings:")
print("\n")
ratings.info()
ratings.head()
print("\n")

print("Movies:")
print("\n")
movies.info()
movies.head()

Keywords:


<class 'pandas.DataFrame'>
RangeIndex: 46419 entries, 0 to 46418
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   id        46419 non-null  int64
 1   keywords  46419 non-null  str  
dtypes: int64(1), str(1)
memory usage: 725.4 KB


Ratings:


<class 'pandas.DataFrame'>
RangeIndex: 100004 entries, 0 to 100003
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100004 non-null  int64  
 1   movieId    100004 non-null  int64  
 2   rating     100004 non-null  float64
 3   timestamp  100004 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


Movies:


<class 'pandas.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  str    
 1   belongs_to_colle

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


### 3. Clean Data and Split into Train/Test


In [4]:
ratings = ratings.dropna(subset=['userId','movieId','rating'])

#users with <5 ratings dropped
counts = ratings['userId'].value_counts()
ratings = ratings[ratings['userId'].isin(counts[counts >= 5].index)].copy()

#clean movies
movies['id'] = pd.to_numeric(movies['id'], errors='coerce')
movies = movies.dropna(subset=['id']).copy()
movies['id'] = movies['id'].astype(int)
movies = movies.drop_duplicates(subset='id')

#clean keywords
keywords['id'] = pd.to_numeric(keywords['id'], errors='coerce')
keywords = keywords.dropna(subset=['id'])
keywords['id'] = keywords['id'].astype(int)

#bridge movielens IDs to TMDB IDs
links['tmdbId'] = pd.to_numeric(links['tmdbId'], errors='coerce')
links = links.dropna(subset=['tmdbId'])
links['movieId'] = links['movieId'].astype(int)
links['tmdbId']  = links['tmdbId'].astype(int)
ml_to_tmdb = dict(zip(links['movieId'], links['tmdbId']))
ratings['tmdb_id'] = ratings['movieId'].map(ml_to_tmdb)
ratings = ratings.dropna(subset=['tmdb_id'])
ratings['tmdb_id'] = ratings['tmdb_id'].astype(int)

#only keep movies with metadata
ratings = ratings[ratings['tmdb_id'].isin(movies['id'])].copy()
print(f"Ratings: {len(ratings):,}")
print(f"Users: {ratings['userId'].nunique():,}")
print(f"Movies: {ratings['tmdb_id'].nunique():,}")

#split into 80/20 (train/test)
ratings.sort_values(['userId', 'timestamp'], inplace=True)

train, test = [], []
for id, group in ratings.groupby('userId'):
    cut = int(len(group) * 0.8)
    train.append(group.iloc[:cut])
    test.append(group.iloc[cut:])

train_df = pd.concat(train).reset_index(drop=True)
test_df  = pd.concat(test).reset_index(drop=True)

print(f"\nTrain: {len(train_df):,} | Test: {len(test_df):,}")

Ratings: 99,810
Users: 671
Movies: 9,025

Train: 79,589 | Test: 20,221


### 4. Index Encoding


In [5]:
all_users  = sorted(ratings['userId'].unique())
all_movies = sorted(ratings['tmdb_id'].unique())

user_to_idx  = {u: i for i, u in enumerate(all_users)}
movie_to_idx = {m: i for i, m in enumerate(all_movies)}

n_users  = len(all_users)
n_movies = len(all_movies)
print(f"n_users = {n_users}")
print(f"n_movies = {n_movies}")

for df in [ratings,train_df,test_df]:
    df['u_idx'] = df['userId'].map(user_to_idx).astype(int)
    df['m_idx'] = df['tmdb_id'].map(movie_to_idx).astype(int)

print(f"Train triples: {len(train_df):,}  |  Test triples: {len(test_df):,}")

n_users = 671
n_movies = 9025
Train triples: 79,589  |  Test triples: 20,221


### 5. Build Content Features (Genres + Keywords)



In [6]:
#parse movie genres into a list
def parse_genres(genres_str):
    try:
        genres = ast.literal_eval(genres_str)
        return [g['name'] for g in genres]
    except:
        return []
movies['genre_list'] = movies['genres'].apply(parse_genres)

#multi-hot encode genres
mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(movies['genre_list'])
genre_df = pd.DataFrame(genre_matrix, index=movies['id'], columns=mlb.classes_)
print(f"Genres ({len(mlb.classes_)}): {list(mlb.classes_)}")

#build keyword strings for movies
kw_dict = dict(zip(keywords['id'], keywords['keywords']))

def get_keywords(tmdb_id):
    try:
        kws = ast.literal_eval(kw_dict.get(tmdb_id, '[]'))
        return ' '.join([k['name'] for k in kws])
    except:
        return ''

movies_active = movies[movies['id'].isin(all_movies)].copy()
movies_active['keywords_string'] = movies_active['id'].apply(get_keywords)

#keywords - tf-idf (keep top 50 features)
tf_idf = TfidfVectorizer(max_features=50, stop_words='english')
kw_matrix = tf_idf.fit_transform(movies_active['keywords_string']).toarray()
kw_df = pd.DataFrame(kw_matrix, index=movies_active['id'])

#combine genre and keyword features into single matrix
genre_active = genre_df.reindex(all_movies).fillna(0)
kw_active = kw_df.reindex(all_movies).fillna(0)

content_features = np.hstack([genre_active.values, kw_active.values]).astype(np.float32)
content_tensor = torch.tensor(content_features, dtype=torch.float32)
content_dim = content_features.shape[1]
print(f"Total content features: {content_dim}")

Genres (20): ['Action', 'Adventure', 'Animation', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'Foreign', 'History', 'Horror', 'Music', 'Mystery', 'Romance', 'Science Fiction', 'TV Movie', 'Thriller', 'War', 'Western']
Total content features: 70


In [7]:
class RatingsDataset(Dataset):
    def __init__(self, df):
        self.u = torch.tensor(df['u_idx'].values, dtype=torch.long)
        self.m = torch.tensor(df['m_idx'].values, dtype=torch.long)
        self.r = torch.tensor(df['rating'].values, dtype=torch.float32)

    def __len__(self):
        return len(self.r)

    def __getitem__(self, idx):
        return self.u[idx], self.m[idx], self.r[idx]

train_loader = DataLoader(RatingsDataset(train_df), batch_size=256, shuffle=True)
test_loader  = DataLoader(RatingsDataset(test_df),  batch_size=256, shuffle=False)

print(f"Train batches: {len(train_loader)} | Test batches: {len(test_loader)}")

Train batches: 311 | Test batches: 79


### 6. NCF Model


In [8]:
class NCF(nn.Module):
    def __init__(self, n_users, n_movies, content_dim, emb_dim=32, content_proj=32, dropout=0.3):
        super().__init__()

        self.user_embeddings = nn.Embedding(n_users, emb_dim)
        self.movie_embeddings = nn.Embedding(n_movies, emb_dim)

        self.content_proj = nn.Sequential(
            nn.Linear(content_dim, content_proj),
            nn.ReLU()
          )

        #MLP - input is user embeddings, movie_embeddings, and projected content
        in_size = emb_dim + emb_dim + content_proj
        self.mlp = nn.Sequential(
            nn.Linear(in_size, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(32, 1)
        )

    def forward(self, user_idx, movie_idx, content_feat):
        u = self.user_embeddings(user_idx)
        m = self.movie_embeddings(movie_idx)
        c = self.content_proj(content_feat)
        x = torch.cat([u, m, c], dim=1)
        return self.mlp(x).squeeze(1)

model = NCF(n_users, n_movies, content_dim).to(DEVICE)

### 7. Train

In [9]:
n_epochs = 30
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

train_rmse_hist = []
test_rmse_hist = []

for epoch in range(n_epochs):
    model.train()
    total_loss = 0
    for u_idx, m_idx, rating in train_loader:
        u_idx, m_idx, rating = u_idx.to(DEVICE), m_idx.to(DEVICE), rating.to(DEVICE)
        content = content_tensor[m_idx].to(DEVICE)

        optimizer.zero_grad()
        preds = model(u_idx, m_idx, content)
        loss = criterion(preds, rating)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    train_rmse = sqrt(total_loss / len(train_loader))
    train_rmse_hist.append(train_rmse)

    # evaluate on test set
    model.eval()
    test_loss = 0
    with torch.no_grad():
        for u_idx, m_idx, rating in test_loader:
            u_idx, m_idx, rating = u_idx.to(DEVICE), m_idx.to(DEVICE), rating.to(DEVICE)
            content = content_tensor[m_idx].to(DEVICE)
            preds = model(u_idx, m_idx, content)
            test_loss += criterion(preds, rating).item()

    test_rmse = sqrt(test_loss / len(test_loader))
    test_rmse_hist.append(test_rmse)

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{n_epochs} - train RMSE: {train_rmse:.4f}, test RMSE: {test_rmse:.4f}")

Epoch 5/30 - train RMSE: 1.1050, test RMSE: 0.9503
Epoch 10/30 - train RMSE: 0.9983, test RMSE: 0.9317
Epoch 15/30 - train RMSE: 0.9201, test RMSE: 0.9207
Epoch 20/30 - train RMSE: 0.8603, test RMSE: 0.9217
Epoch 25/30 - train RMSE: 0.8191, test RMSE: 0.9308
Epoch 30/30 - train RMSE: 0.7927, test RMSE: 0.9411


### 8. Evaluation

In [10]:
all_preds = []
all_true = []

model.eval()
with torch.no_grad():
    for u_idx, m_idx, rating in test_loader:
        preds = model(u_idx, m_idx, content_tensor[m_idx])
        all_preds.extend(preds.numpy())
        all_true.extend(rating.numpy())

all_preds = np.array(all_preds)
all_true = np.array(all_true)
test_rmse = sqrt(mean_squared_error(all_true, all_preds))
test_mae = np.mean(np.abs(all_true - all_preds))

print(f"Test RMSE: {test_rmse:.4f}")
print(f"Test MAE: {test_mae:.4f}")

Test RMSE: 0.9411
Test MAE: 0.7155


### 9. Top-N Recommendations

In [11]:
#mappings
tmdb_to_title = dict(zip(movies['id'], movies['title']))
idx_to_movie = {v: k for k, v in movie_to_idx.items()}

def recommend(user_raw_id, n=10):
    u_idx = user_to_idx[user_raw_id]
    seen = set(ratings[ratings['userId'] == user_raw_id]['tmdb_id'])
    candidate_indexes = [movie_to_idx[m] for m in all_movies if m not in seen]
    model.eval()

    with torch.no_grad():
        u_tensor = torch.tensor([u_idx] * len(candidate_indexes),dtype=torch.long)
        m_tensor = torch.tensor(candidate_indexes,dtype=torch.long)
        scores = model(u_tensor,m_tensor,content_tensor[m_tensor]).numpy()

    top_indexes = np.argsort(scores)[::-1][:n]
    results = []
    for i in top_indexes:
        tmdb_id = idx_to_movie[candidate_indexes[i]]
        title = tmdb_to_title.get(tmdb_id, 'Unknown')
        results.append((title, round(float(scores[i]), 3)))
    return results

sample_user = all_users[0]
print(f"Top 10 recommendations for user {sample_user}:")
for rank, (title, score) in enumerate(recommend(sample_user,n=10),1):
    print(f"{rank:>2}. Movie: {title} | Predicted rating: {score}")

Top 10 recommendations for user 1:
 1. Movie: Salesman | Predicted rating: 4.281
 2. Movie: Diabolique | Predicted rating: 4.268
 3. Movie: Code Unknown | Predicted rating: 4.25
 4. Movie: Mirror | Predicted rating: 4.22
 5. Movie: Lake of Fire | Predicted rating: 4.203
 6. Movie: Character | Predicted rating: 4.171
 7. Movie: Pumpkinhead | Predicted rating: 4.155
 8. Movie: Maelström | Predicted rating: 4.152
 9. Movie: Rosetta | Predicted rating: 4.145
10. Movie: The Book Thief | Predicted rating: 4.141


### 10. Model Comparison

In [12]:
import random

def evaluate_ncf(model, train_df, test_df, user_to_idx, movie_to_idx, k=10, n_candidates=100, seed=42):
    random.seed(seed)
    np.random.seed(seed)

    all_movie_ids = set(movie_to_idx.keys())
    train_seen_by_user = train_df.groupby("userId")["tmdb_id"].apply(set).to_dict()
    test_pos_by_user = test_df.groupby("userId")["tmdb_id"].apply(list).to_dict()

    hits, ndcgs = [], []
    users = [u for u in test_pos_by_user.keys() if u in user_to_idx]

    for user_id in users:
        positives = test_pos_by_user.get(user_id, [])
        if not positives:
            continue

        watched_movie = random.choice(positives)

        excluded = train_seen_by_user.get(user_id, set()).union(set(positives))
        not_watched = list(all_movie_ids - excluded)
        if len(not_watched) < n_candidates - 1:
            continue

        negative_sample = random.sample(not_watched, n_candidates - 1)
        candidates = [watched_movie] + negative_sample
        random.shuffle(candidates)

        u_idx = user_to_idx[user_id]
        candidate_indexes = [movie_to_idx[m] for m in candidates if m in movie_to_idx]

        model.eval()
        with torch.no_grad():
            u_tensor = torch.tensor([u_idx] * len(candidate_indexes),dtype=torch.long)
            m_tensor = torch.tensor(candidate_indexes,dtype=torch.long)
            scores = model(u_tensor,m_tensor,content_tensor[m_tensor]).numpy()

        ranked_ids = [candidate_indexes[i] for i in np.argsort(scores)[::-1]]
        relevance = [int(m == movie_to_idx.get(watched_movie)) for m in ranked_ids]

        hits.append(int(movie_to_idx.get(watched_movie) in ranked_ids[:k]))
        ndcg_score = 1 / np.log2(relevance.index(1) + 2) if 1 in relevance else 0.0
        ndcgs.append(ndcg_score)

    return {
        "hit_rate": float(np.mean(hits)) if hits else 0.0,
        "ndcg": float(np.mean(ndcgs)) if ndcgs else 0.0,
        "n_users": len(hits),
    }

# run evaluation
eval_results = evaluate_ncf(
    model=model,
    train_df=train_df,
    test_df=test_df,
    user_to_idx=user_to_idx,
    movie_to_idx=movie_to_idx,
    k=10,
    n_candidates=100,
    seed=42,
)
print("NCF")
print(f"Users evaluated: {eval_results['n_users']}")
print(f"HitRate@10: {eval_results['hit_rate']:.4f}")
print(f"NDCG@10: {eval_results['ndcg']:.4f}")

NCF
Users evaluated: 671
HitRate@10: 0.1356
NDCG@10: 0.2181
